In [1]:
import scipy.stats as stats
from plotly.io import show
from sklearn import set_config
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline

from skfolio import Population, RatioMeasure, RiskMeasure
from skfolio.datasets import load_ftse100_dataset
from skfolio.metrics import make_scorer
from skfolio.model_selection import MultipleRandomizedCV, WalkForward, cross_val_predict
from skfolio.moments import ShrunkCovariance
from skfolio.optimization import HierarchicalEqualRiskContribution
from skfolio.pre_selection import SelectKExtremes
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EmpiricalPrior

set_config(transform_output="pandas")

prices = load_ftse100_dataset()
returns = prices_to_returns(prices)

# Sequential train-test split: 67% training, 33% testing.
# `shuffle=False` preserves chronological order, crucial for time-series data.
X_train, X_test = train_test_split(returns, test_size=0.33, shuffle=False)

In [2]:
pre_selection = SelectKExtremes(k=10, measure=RatioMeasure.SHARPE_RATIO, highest=True)

optimization = HierarchicalEqualRiskContribution(
    prior_estimator=EmpiricalPrior(
        covariance_estimator=ShrunkCovariance(shrinkage=0.5)
    ),
    risk_measure=RiskMeasure.VARIANCE,
)

model_bench = Pipeline(
    [
        ("pre_selection", pre_selection),
        ("optimization", optimization),
    ]
)

In [3]:

walk_forward = WalkForward(test_size=20, train_size=252)

In [4]:
random_search = RandomizedSearchCV(
    estimator=model_bench,
    cv=walk_forward,
    n_jobs=-1,
    param_distributions={
        "pre_selection__k": stats.randint(10, 30),
        "optimization__prior_estimator__covariance_estimator__shrinkage": stats.uniform(
            0, 1
        ),
    },
    n_iter=30,
    random_state=0,
    scoring=make_scorer(RatioMeasure.CVAR_RATIO),
)
random_search.fit(X_train)

# Retrieve the best estimator from the search.
model_tuned = random_search.best_estimator_
model_tuned

,steps,"[('pre_selection', ...), ('optimization', ...)]"
,transform_input,None
,memory,None
,verbose,False
,k,27
,measure,Sharpe Ratio
,highest,True
,risk_measure,Variance
,prior_estimator,EmpiricalPrio...138825158765))
,distance_estimator,None
,hierarchical_clustering_estimator,None


In [5]:
pred_bench = cross_val_predict(model_bench, X_test, cv=walk_forward)
pred_bench.name = "Benchmark Model"

pred_tuned = cross_val_predict(model_tuned, X_test, cv=walk_forward, n_jobs=-1)
pred_tuned.name = "Tuned Model"

# Combine results for easier analysis.
population = Population([pred_bench, pred_tuned])
population.plot_cumulative_returns()

In [7]:
population.summary()

,Benchmark Model,Tuned Model
Mean,0.022%,0.041%
Annualized Mean,5.52%,10.39%
Variance,0.013%,0.010%
Annualized Variance,3.33%,2.60%
Semi-Variance,0.0072%,0.0056%
Annualized Semi-Variance,1.81%,1.42%
Standard Deviation,1.15%,1.02%
Annualized Standard Deviation,18.26%,16.13%
Semi-Deviation,0.85%,0.75%
Annualized Semi-Deviation,13.44%,11.92%


In [8]:
cv_mc = MultipleRandomizedCV(
    walk_forward=walk_forward,
    n_subsamples=500,
    asset_subset_size=50,
    window_size=3 * 252,
    random_state=0,
)

# Generate cross-validated predictions for both models.
pred_bench_mc = cross_val_predict(
    model_bench,
    X_test,
    cv=cv_mc,
    n_jobs=-1,
    portfolio_params={"tag": "Benchmark Model"},
)

pred_tuned_mc = cross_val_predict(
    model_tuned, X_test, cv=cv_mc, n_jobs=-1, portfolio_params={"tag": "Tuned Model"}
)

# Combine results for easier analysis.
population_mc = pred_bench_mc + pred_tuned_mc

In [9]:
fig = pred_tuned_mc[:10].plot_cumulative_returns(use_tag_in_legend=False)
show(fig)

In [10]:
population_mc.plot_distribution(
    measure_list=[RatioMeasure.ANNUALIZED_SHARPE_RATIO],
    tag_list=["Benchmark Model", "Tuned Model"],
)

In [11]:
for pred in [pred_bench_mc, pred_tuned_mc]:
    tag = pred[0].tag
    mean_sr = pred.measures_mean(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    std_sr = pred.measures_std(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO)
    print(f"{tag}\n{'=' * len(tag)}")
    print(f"Average Sharpe Ratio: {mean_sr:0.2f}")
    print(f"Sharpe Ratio Std Dev: {std_sr:0.2f}\n")

Benchmark Model
Average Sharpe Ratio: 0.34
Sharpe Ratio Std Dev: 0.36

Tuned Model
Average Sharpe Ratio: 0.53
Sharpe Ratio Std Dev: 0.34



In [12]:
population_mc.boxplot_measure(
    measure=RatioMeasure.CVAR_RATIO, tag_list=["Benchmark Model", "Tuned Model"]
)

In [13]:
pred_tuned_mc[:2].plot_composition(display_sub_ptf_name=False)

In [14]:
pred_tuned_mc[0].plot_weights_per_observation()

A single-path walk-forward analysis may understate the variability and uncertainty of real-world performance. Multiple Randomized Cross-Validation, by contrast, applies a resampling-based evaluation across asset subsets and time windows, yielding performance estimates that are more robust and less prone to overfitting.

References